# Integrated: MLP

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

class IntegratedMLP(nn.Module):
    def __init__(self, alpha_init=0.1, beta_init=0.85, omega_init=0.01):
        super(IntegratedMLP, self).__init__()
        self.alpha = nn.Parameter(torch.tensor([alpha_init]))
        self.beta = nn.Parameter(torch.tensor([beta_init]))
        self.omega_raw = nn.Parameter(torch.tensor([omega_init]).log().exp())

    def forward(self, eps_sq_tm1, sigma_sq_tm1):
        omega = torch.nn.functional.softplus(self.omega_raw)
        sigma_sq = omega + self.alpha * eps_sq_tm1 + self.beta * sigma_sq_tm1
        return torch.nn.functional.softplus(sigma_sq)


def negative_log_likelihood(y, sigma_sq):
    return 0.5 * torch.mean(torch.log(sigma_sq) + y**2 / sigma_sq)


def train_integrated_MLP(returns_tensor, dates=None, train_ratio=0.8, num_epochs=200, lr=0.001, warmup_window=20):
    T = len(returns_tensor)
    T_train = int(T * train_ratio)
    T_test = T - T_train

    mean = returns_tensor[:T_train].mean()
    std = returns_tensor[:T_train].std()
    returns_norm = (returns_tensor - mean) / std
    train_returns = returns_norm[:T_train]
    test_returns = returns_norm[T_train:]

    initial_var = torch.var(train_returns[:warmup_window])
    omega_guess = initial_var * (1 - 0.1 - 0.85)
    model = IntegratedMLP(omega_init=omega_guess)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    loss_history = []

    for epoch in range(num_epochs):
        sigma_sq_list = []
        sigma_sq_prev = initial_var

        for t in range(T_train):
            if t == 0:
                sigma_sq_list.append(sigma_sq_prev)
                continue
            eps_sq_tm1 = train_returns[t - 1].view(1, 1) ** 2
            sigma_sq_tm1 = sigma_sq_prev.view(1, 1)
            sigma_sq_t = model(eps_sq_tm1, sigma_sq_tm1).squeeze()
            sigma_sq_list.append(sigma_sq_t)
            sigma_sq_prev = sigma_sq_t

        sigma_sq_train = torch.stack(sigma_sq_list)
        loss = negative_log_likelihood(train_returns[warmup_window:], sigma_sq_train[warmup_window:])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_history.append(loss.item())
        #if epoch % 10 == 0:
        print(f"Epoch {epoch}, Train NLL: {loss.item():.6f}")

    sigma_sq_list = []
    sigma_sq_prev = sigma_sq_train[-1].detach()

    for t in range(T_test):
        if t == 0:
            sigma_sq_list.append(sigma_sq_prev)
            continue
        eps_sq_tm1 = test_returns[t - 1].view(1, 1) ** 2
        sigma_sq_tm1 = sigma_sq_prev.view(1, 1)
        sigma_sq_t = model(eps_sq_tm1, sigma_sq_tm1).squeeze()
        sigma_sq_list.append(sigma_sq_t)
        sigma_sq_prev = sigma_sq_t

    sigma_sq_test = torch.stack(sigma_sq_list)
    sigma_sq_test_rescaled = sigma_sq_test * std**2
    test_returns_rescaled = returns_tensor[T_train:]
    test_nll = negative_log_likelihood(test_returns_rescaled[warmup_window:], sigma_sq_test_rescaled[warmup_window:])
    print(f"\nTest NLL: {test_nll.item():.6f}")

    if dates is not None:
        test_dates = dates[T_train:]
        x_values = test_dates[warmup_window:]
    else:
        x_values = list(range(T_test - warmup_window))
    return(sigma_sq_test_rescaled,test_returns_rescaled, model, x_values,loss_history)

In [ ]:
@torch.no_grad()
def _filter_sigma2_over(series_norm: torch.Tensor,
                        model: torch.nn.Module,
                        initial_var: torch.Tensor) -> torch.Tensor:
    T = len(series_norm)
    out = []
    sigma_prev = initial_var
    for t in range(T):
        if t == 0:
            out.append(sigma_prev)
        else:
            eps_sq_tm1 = series_norm[t-1].view(1,1)**2
            sigma_sq_tm1 = sigma_prev.view(1,1)
            sigma_t = model(eps_sq_tm1, sigma_sq_tm1).squeeze()
            out.append(sigma_t)
            sigma_prev = sigma_t
    return torch.stack(out)


@torch.no_grad()
def forecast_h_steps(model: torch.nn.Module,
                     returns_tensor: torch.Tensor,
                     train_ratio: float = 0.8,
                     H: int = 20,
                     warmup_window: int = 20):
    
    T = len(returns_tensor)
    T_train = int(T * train_ratio)

    mean = returns_tensor[:T_train].mean()
    std  = returns_tensor[:T_train].std()
    series_norm = (returns_tensor - mean) / std

    initial_var = torch.var(series_norm[:warmup_window])
    sigma_seq = _filter_sigma2_over(series_norm[:T_train], model, initial_var)
    sigma_last = sigma_seq[-1]                      
    eps_last_sq = series_norm[T_train-1]**2     

    sigma_fore = []
    sigma_prev = sigma_last
    eps_sq_prev = eps_last_sq
    for h in range(1, H+1):
        sigma_h = model(eps_sq_prev.view(1,1), sigma_prev.view(1,1)).squeeze()
        sigma_fore.append(sigma_h)
        eps_sq_prev = sigma_h
        sigma_prev = sigma_h

    sigma2_forecasts = torch.stack(sigma_fore)
    sigma2_forecasts_rescaled = sigma2_forecasts * (std**2)
    sigma2_last_rescaled = sigma_last * (std**2)
    return sigma2_forecasts_rescaled, sigma2_last_rescaled


@torch.no_grad()
def lower_triangular_forecasts(model: torch.nn.Module,
                               returns_tensor: torch.Tensor,
                               train_ratio: float = 0.8,
                               H: int = 20,
                               warmup_window: int = 20):
    T = len(returns_tensor)
    T_train = int(T * train_ratio)

    mean = returns_tensor[:T_train].mean()
    std  = returns_tensor[:T_train].std()
    series_norm = (returns_tensor - mean) / std

    initial_var = torch.var(series_norm[:warmup_window])
    sigma_all = _filter_sigma2_over(series_norm, model, initial_var)

    test_norm = series_norm[T_train:] #series_norm[T_train:forecast_horizon]
    origins = len(test_norm)
    F = torch.zeros((origins, H), dtype=series_norm.dtype)

    for i in range(origins): 
        t_abs = T_train + i
        eps_sq_prev = series_norm[t_abs]**2
        sigma_prev = sigma_all[t_abs]

        for h in range(1, H+1):
            sigma_h = model(eps_sq_prev.view(1,1), sigma_prev.view(1,1)).squeeze()
            F[i, h-1] = sigma_h
            # recurse
            eps_sq_prev = sigma_h
            sigma_prev = sigma_h

    F_rescaled = F * (std**2)
    origins_idx = torch.arange(T_train, T) 
    return F_rescaled, origins_idx

# Integrated: LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class IntegratedLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_f = nn.Linear(input_size, hidden_size)
        self.U_f = nn.Linear(1, hidden_size, bias=False)

        self.W_i = nn.Linear(input_size, hidden_size)
        self.U_i = nn.Linear(1, hidden_size, bias=False)

        self.W_c = nn.Linear(input_size, hidden_size)
        self.U_c = nn.Linear(1, hidden_size, bias=False)

        self.w = nn.Parameter(torch.tensor([0.1]))

        self.alpha = nn.Parameter(torch.tensor([0.1]))
        self.beta = nn.Parameter(torch.tensor([0.85]))
        self.omega_raw = nn.Parameter(torch.tensor([0.01]).log().exp())
        
    def forward(self, eps_sq_tm1, sigma_sq_tm1, c_tm1):
        f_t = torch.sigmoid(self.W_f(eps_sq_tm1) + self.U_f(sigma_sq_tm1))
        i_t = torch.sigmoid(self.W_i(eps_sq_tm1) + self.U_i(sigma_sq_tm1))
        c_tilde = torch.tanh(self.W_c(eps_sq_tm1) + self.U_c(sigma_sq_tm1))

        c_t = f_t * c_tm1 + i_t * c_tilde

        omega = F.softplus(self.omega_raw)
        o_t = omega + self.alpha * eps_sq_tm1 + self.beta * sigma_sq_tm1

        sigma_sq_t = o_t * (1 + self.w * torch.tanh(c_t).mean(dim=1, keepdim=True))

        return sigma_sq_t, c_t

In [ ]:
@torch.no_grad()
def lower_triangular_from_test_lstm(model,
                                    model_name,
                                    test_sq_returns: torch.Tensor,
                                    sigma_sq_init: torch.Tensor,
                                    c_init: torch.Tensor,
                                    clamp_min: float = 1e-12,
                                    lower_triangular = False):
    device = next(model.parameters()).device
    x = test_sq_returns.to(device).to(torch.float32)   
    sigma_prev = sigma_sq_init.to(device).to(torch.float32)  
    c_prev = c_init.to(device).to(torch.float32)
    T = len(x)#forecast_horizon # 

    sigma_filt = torch.zeros(T, 1, device=device)
    c_states   = torch.zeros(T, c_prev.shape[1], device=device)

    sigma_filt[0:1] = sigma_prev
    c_states[0:1]   = c_prev
    for i in range(1, T):
        eps_sq_tm1 = x[i-1].view(1,1)
        sigma_i, c_prev = model(eps_sq_tm1, sigma_prev, c_prev)
        sigma_i = torch.clamp(sigma_i, min=clamp_min)
        sigma_filt[i:i+1] = sigma_i
        c_states[i:i+1]   = c_prev
        sigma_prev = sigma_i.detach()

    F = torch.full((T, T), float('nan'), device=device)
    for i in range(T):
        eps_sq_prev = x[i].view(1,1)             
        sigma_prev  = sigma_filt[i].view(1,1)   
        c_prev      = c_states[i].view(1, -1)
        
        if lower_triangular == True:
            max_h = T - i
            
        else:
            max_h = T
        for h in range(1, max_h + 1):
            sigma_h, c_prev = model(eps_sq_prev, sigma_prev, c_prev)
            sigma_h = torch.clamp(sigma_h, min=clamp_min)
            F[i, h-1] = sigma_h.squeeze()
            eps_sq_prev = sigma_h
            sigma_prev  = sigma_h
        
        print(#"####################################\n",
              "Forecast Origin:",
              i,
              "\n####################################")

    cols = [f"h.{k:03d}" for k in range(1, T + 1)]
    n_step_preds_matrix = pd.DataFrame(F, columns=cols)#range(1200,1200 + forecast_horizon), columns=cols)
    
    n_step_qlikes = []
    for i in range(forecast_horizon):
        df_h_step = pd.concat([n_step_preds_matrix.iloc[:,i].dropna(),pd.Series(test_sq_returns)],axis=1).dropna()
                                                                                #,index = range(1200,1200 + forecast_horizon))],axis=1).dropna()#index = range(1200,1500))],axis=1).dropna()

        n_step_qlikes.append(forecast_metrics(preds = df_h_step.iloc[:,0],
                         y_test = df_h_step.iloc[:,1],
                         model = model_name,
                         qlike_only = True))#[-1])
    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv",header = False, index = False)

    return df_qlikes, n_step_preds_matrix